In [ ]:
# --- Setup: dataset onto Colab's local disk (run once per session) ---
import os, sys, time, zipfile

DATA = "/content/data"
if not os.path.isdir(DATA):
    from google.colab import drive
    drive.mount("/content/drive")
    t = time.time()
    with zipfile.ZipFile("/content/drive/MyDrive/data.zip") as z:
        z.extractall("/content")
    print(f"unzipped to {DATA} in {time.time()-t:.0f}s")

sys.path.append(f"{DATA}/scripts")
from load_dataset import Dataset

ds = Dataset(DATA)
print(f"camera: {len(ds.camera)} frames | gt: {len(ds.gt)} poses | "
      f"span: {(ds.gt.t_ns.iloc[-1] - ds.gt.t_ns.iloc[0]) / 1e9:.1f} s | "
      f"images on disk: {len(os.listdir(f'{DATA}/camera/images'))}")

# 1 · The experiment at a glance

In [ ]:
# Where: GT trajectory on the lidar map vs raw wheel odometry
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

mp = np.load(f"{DATA}/ground_truth/map_points.npz")["xy"]
tg = (ds.gt.t_ns - ds.gt.t_ns.iloc[0]) / 1e9

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(*mp.T, s=0.6, c="0.3", lw=0, rasterized=True)
ax.plot(ds.odom.x, ds.odom.y, "--", c="crimson", lw=1.3, alpha=0.75,
        label="wheel dead reckoning")
sc = ax.scatter(ds.gt.x, ds.gt.y, c=tg, s=5, cmap="viridis", lw=0, zorder=3,
                label="ground truth (lidar)")
ax.plot([ds.gt.x.iloc[-1], ds.odom.x.iloc[-1]], [ds.gt.y.iloc[-1], ds.odom.y.iloc[-1]],
        c="crimson", lw=1, zorder=4)
gap = np.hypot(ds.gt.x.iloc[-1] - ds.odom.x.iloc[-1], ds.gt.y.iloc[-1] - ds.odom.y.iloc[-1])
ax.annotate(f"odometry ends {gap*100:.0f} cm off", xy=(ds.odom.x.iloc[-1], ds.odom.y.iloc[-1]),
            xytext=(16, -22), textcoords="offset points", color="crimson", fontsize=9,
            arrowprops=dict(arrowstyle="-", color="crimson", lw=0.7))
ax.plot(ds.gt.x.iloc[0], ds.gt.y.iloc[0], "o", ms=9, mfc="none", mec="k", mew=1.5, zorder=5)
ax.annotate("start = end (loop closes)", xy=(ds.gt.x.iloc[0], ds.gt.y.iloc[0]),
            xytext=(-8, 14), textcoords="offset points", fontsize=9, ha="right")
ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_xlim(ds.gt.x.min() - 1.5, ds.gt.x.max() + 1.6)
ax.set_ylim(ds.gt.y.min() - 1.3, ds.gt.y.max() + 1.5)
ax.legend(loc="upper left", frameon=False, fontsize=9, markerscale=2.5)
fig.colorbar(sc, ax=ax, label="time [s]", shrink=0.8)
ax.set_title("One 17.8 m lap, 117 s — ground truth vs wheel dead reckoning")
plt.show()

In [ ]:
# When: five async streams on one clock axis
t0 = int(ds.camera.t_ns.iloc[0])
sec = lambda s: (np.asarray(s, dtype=np.int64) - t0) / 1e9
cam_t, imu_t, odo_t, gt_t = sec(ds.camera.t_ns), sec(ds.imu.t_ns), sec(ds.odom.t_ns), sec(ds.gt.t_ns)
wf = ds.wifi.groupby("scan_idx").agg(a=("t_start_ns", "first"), b=("t_end_ns", "first"))
w0, w1 = sec(wf.a), sec(wf.b)

v = np.hypot(np.diff(ds.gt.x), np.diff(ds.gt.y)) / np.diff(gt_t)
mv = np.convolve(v, np.ones(9) / 9, "same") > 0.03          # smoothed |v| vs GT noise floor
tA, tB = gt_t[np.argmax(mv)], gt_t[len(mv) - np.argmax(mv[::-1])]

rows = [("camera  ~30 Hz", cam_t, "#4c72b0"), ("imu  20 Hz", imu_t, "#dd8452"),
        ("wheel odom  20 Hz", odo_t, "#55a868"), ("wifi  28 scans", None, "#8172b3"),
        ("lidar → GT  ~8.6 Hz", gt_t, "#c44e52")]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.6), height_ratios=[2.1, 1],
                             constrained_layout=True)
Z0, Z1 = 60.0, 61.2
for ax, lw_ in ((a1, 0.25), (a2, 1.2)):
    for i, (name, tt, c) in enumerate(rows):
        y = len(rows) - 1 - i
        if tt is None:
            ax.broken_barh(list(zip(w0, w1 - w0)), (y - 0.3, 0.6), color=c, alpha=0.55, lw=0)
        else:
            ax.eventplot(tt, lineoffsets=y, linelengths=0.6, colors=c, lw=lw_)
    ax.set_yticks(range(len(rows)), [r[0] for r in rows][::-1], fontsize=9)
    ax.set_ylim(-0.55, len(rows) - 0.45)
a1.set_ylim(-1.0, len(rows) - 0.45)
for (ta, tb, lab) in ((-2.6, tA, "stationary"), (tB, 118, "stationary")):
    a1.axvspan(ta, tb, color="0.55", alpha=0.18, lw=0)
    a1.text((max(ta, -2.6) + min(tb, 118)) / 2, -0.62, lab, ha="center", va="top",
            fontsize=8, color="0.35")
a1.axvspan(Z0, Z1, color="gold", alpha=0.35, lw=0)
a1.set_xlim(-2.6, 118)
a1.set_title("Five sensor streams, four independent clocks — raw asynchronous timeline")
a2.set_xlim(Z0, Z1)
a2.set_xlabel("time since first camera frame [s]")
a2.set_title("zoom: 1.2 s while driving — only imu & wheel odom share a tick; "
             "the rest never align", fontsize=9)
plt.show()